# ⚖️ Playground · Balanceo de clases

> ⚠️ Este notebook es **autónomo**: **no** importa nada de `src/`. Todo el código vive aquí, igual que en los notebooks de `recursos/`. Es un cuaderno de exploración libre que recoge lo que aprendimos practicando aquí.

## 0. Configuración del Notebook

Importamos todas las librerías que usaremos. Trabajaremos con dos modelos (regresión logística y XGBoost) y tres estrategias frente al desbalance: *baseline*, ponderación de clases (`class_weight` / `scale_pos_weight`) y sobremuestreo sintético (**SMOTE**).

In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = 'notebook_connected'

import warnings; warnings.filterwarnings('ignore')

from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
)
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

### Constantes y funciones auxiliares

In [2]:
# Datasets YA procesados por el notebook 02 (train/test separados, fit-on-train)
PATH_DIRECTORIO_DATOS = "../../data/processed"
PATH_TRAIN = f"{PATH_DIRECTORIO_DATOS}/train.csv"
PATH_TEST = f"{PATH_DIRECTORIO_DATOS}/test.csv"

target_column = 'is_canceled'
dict_target_hotel = {0: 'No cancelada', 1: 'Cancelada'}

In [3]:
def evaluar(y_true, y_pred, y_proba):
    """Calcula las métricas de clasificación más relevantes para este problema.

    Args:
        y_true: etiquetas reales (0/1).
        y_pred: etiquetas predichas (0/1) con el umbral por defecto (0.5).
        y_proba: probabilidad estimada de la clase positiva (cancelación).

    Returns:
        dict con accuracy, precision, recall, f1 y roc_auc.
    """
    return {
        'accuracy':  accuracy_score(y_true, y_pred),
        'precision': precision_score(y_true, y_pred),
        'recall':    recall_score(y_true, y_pred),
        'f1':        f1_score(y_true, y_pred),
        'roc_auc':   roc_auc_score(y_true, y_proba),
    }

## 1. El problema del desbalance

En este dataset **alrededor del 37 % de las reservas se cancelan** y el 63 % no. No es un desbalance extremo, pero sí suficiente para que algunas decisiones importen.

¿Por qué nos preocupa?

- **El *accuracy* engaña.** Un modelo tonto que prediga *siempre* "No cancelada" acertaría un ~63 % — parece "bien", pero **no detecta ni una sola cancelación**, que es justo lo que nos interesa.
- Por eso miramos sobre todo el **recall** de la clase positiva (de todas las cancelaciones reales, ¿cuántas pillamos?) y el **ROC-AUC** (calidad del *ranking* de probabilidades, **independiente del umbral**).

Vamos a comparar tres estrategias clásicas para "compensar" el desbalance y ver qué le pasa a cada métrica.

In [4]:
# Reparto de clases sobre el dataset ya procesado (train + test, solo para visualizar)
y_full = pd.concat([pd.read_csv(PATH_TRAIN)[target_column],
                    pd.read_csv(PATH_TEST)[target_column]])

conteo_abs = y_full.value_counts().sort_index()
conteo_rel = y_full.value_counts(normalize=True).sort_index()

resumen_clases = pd.DataFrame({
    'clase': [dict_target_hotel[i] for i in conteo_abs.index],
    'reservas': conteo_abs.values,
    'proporcion': conteo_rel.values,
})
resumen_clases

,clase,reservas,proporcion
0,No cancelada,74388,0.627413
1,Cancelada,44175,0.372587


In [5]:
# Visualizamos el desbalance
fig = px.bar(
    resumen_clases,
    x='clase',
    y='reservas',
    text=resumen_clases['proporcion'].map(lambda p: f'{p:.1%}'),
    color='clase',
    title='Reparto de clases: ~37 % de las reservas se cancelan',
    labels={'clase': 'Clase', 'reservas': 'Número de reservas'},
    width=650, height=420,
)
fig.update_traces(textposition='outside')
fig.show(renderer='notebook_connected')

## 2. Datos

Cargamos el dataset ya preparado por [`02_preparacion_datos.ipynb`](02_preparacion_datos.ipynb): limpio, codificado (*one-hot*) y separado en `train`/`test` (preprocesado **fit-on-train**). 

> ⚠️ El balanceo (ponderación o SMOTE) se aplica **solo al `train`** —nunca al `test`—: el conjunto de prueba debe conservar la proporción **real** de cancelaciones (~37 %), que es la que veremos en producción. Por eso aquí basta con cargar; la partición ya viene hecha.

In [6]:
train_df = pd.read_csv(PATH_TRAIN)
test_df = pd.read_csv(PATH_TEST)

X_train = train_df.drop(columns=target_column)
y_train = train_df[target_column]
X_test = test_df.drop(columns=target_column)
y_test = test_df[target_column]
assert list(X_train.columns) == list(X_test.columns), "Columnas train/test no coinciden"

print('Train:', X_train.shape, '| Test:', X_test.shape)
print('\nDistribución en train:')
print(y_train.value_counts(normalize=True).rename(index=dict_target_hotel).round(4))

Train: (94850, 155) | Test: (23713, 155)

Distribución en train:
is_canceled
No cancelada    0.6274
Cancelada       0.3726
Name: proportion, dtype: float64


Guardamos aquí la lista donde iremos acumulando los resultados de cada (modelo, estrategia) para la comparativa final.

In [7]:
# Acumulador de resultados
resultados = []

# Conteos de clases en TRAIN (los necesitaremos para scale_pos_weight)
n_neg = int((y_train == 0).sum())
n_pos = int((y_train == 1).sum())
ratio_neg_pos = n_neg / n_pos
print(f'Negativos (no cancela): {n_neg}')
print(f'Positivos (cancela):    {n_pos}')
print(f'Ratio neg/pos:          {ratio_neg_pos:.3f}')

Negativos (no cancela): 59510
Positivos (cancela):    35340
Ratio neg/pos:          1.684


## 3. Estrategia A · Baseline (sin balanceo)

Entrenamos los dos modelos **tal cual**, sin ninguna corrección del desbalance. Será nuestra referencia.

In [8]:
# --- Regresión logística (baseline) ---
logreg_base = LogisticRegression(max_iter=1000, random_state=42)
logreg_base.fit(X_train, y_train)

y_pred = logreg_base.predict(X_test)
y_proba = logreg_base.predict_proba(X_test)[:, 1]

m = evaluar(y_test, y_pred, y_proba)
m.update({'modelo': 'LogisticRegression', 'estrategia': 'A · Baseline'})
resultados.append(m)
m

{'accuracy': 0.8118753426390587,
 'precision': 0.8067321178120617,
 'recall': 0.6510469722693831,
 'f1': 0.7205762605699969,
 'roc_auc': 0.8916053283171721,
 'modelo': 'LogisticRegression',
 'estrategia': 'A · Baseline'}

In [9]:
# --- XGBoost (baseline) ---
xgb_base = XGBClassifier(
    n_estimators=300, max_depth=8, learning_rate=0.1,
    eval_metric='logloss', n_jobs=-1, random_state=42,
)
xgb_base.fit(X_train, y_train)

y_pred = xgb_base.predict(X_test)
y_proba = xgb_base.predict_proba(X_test)[:, 1]

m = evaluar(y_test, y_pred, y_proba)
m.update({'modelo': 'XGBoost', 'estrategia': 'A · Baseline'})
resultados.append(m)
m

{'accuracy': 0.8737401425378485,
 'precision': 0.8575976490755479,
 'recall': 0.7927560837577815,
 'f1': 0.8239030702270321,
 'roc_auc': 0.9467216515111438,
 'modelo': 'XGBoost',
 'estrategia': 'A · Baseline'}

## 4. Estrategia B · Ponderación de clases

En lugar de tocar los datos, le decimos al modelo que **pondere más los errores sobre la clase minoritaria**:

- Regresión logística → `class_weight='balanced'` (pesos inversamente proporcionales a la frecuencia de cada clase).
- XGBoost → `scale_pos_weight = nº negativos / nº positivos` (≈ 1.7 aquí).

No cambia el dataset, solo la **función de pérdida**.

In [10]:
# --- Regresión logística con class_weight='balanced' ---
logreg_w = LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced')
logreg_w.fit(X_train, y_train)

y_pred = logreg_w.predict(X_test)
y_proba = logreg_w.predict_proba(X_test)[:, 1]

m = evaluar(y_test, y_pred, y_proba)
m.update({'modelo': 'LogisticRegression', 'estrategia': 'B · class_weight'})
resultados.append(m)
m

{'accuracy': 0.809007717285877,
 'precision': 0.727013918177984,
 'recall': 0.7804187889077533,
 'f1': 0.7527703477264043,
 'roc_auc': 0.8916912411857147,
 'modelo': 'LogisticRegression',
 'estrategia': 'B · class_weight'}

In [11]:
# --- XGBoost con scale_pos_weight = ratio neg/pos ---
xgb_w = XGBClassifier(
    n_estimators=300, max_depth=8, learning_rate=0.1,
    eval_metric='logloss', n_jobs=-1, random_state=42,
    scale_pos_weight=ratio_neg_pos,
)
xgb_w.fit(X_train, y_train)

y_pred = xgb_w.predict(X_test)
y_proba = xgb_w.predict_proba(X_test)[:, 1]

m = evaluar(y_test, y_pred, y_proba)
m.update({'modelo': 'XGBoost', 'estrategia': 'B · scale_pos_weight'})
resultados.append(m)
m

{'accuracy': 0.8695652173913043,
 'precision': 0.8097087378640777,
 'recall': 0.8495755517826825,
 'f1': 0.8291632145816072,
 'roc_auc': 0.9466994410604476,
 'modelo': 'XGBoost',
 'estrategia': 'B · scale_pos_weight'}

## 5. Estrategia C · SMOTE (sobremuestreo sintético)

**SMOTE** genera ejemplos sintéticos de la clase minoritaria interpolando entre vecinos cercanos, hasta igualar las clases.

⚠️ **Clave metodológica:** SMOTE debe aplicarse **solo a los datos de entrenamiento**, nunca al *test* (sería hacer trampa). Lo garantizamos metiéndolo dentro de un `Pipeline` de `imblearn`: en `fit` remuestrea solo *train*, y en `predict` los datos pasan directos al modelo sin tocarse.

In [12]:
# --- Regresión logística + SMOTE ---
pipe_logreg = ImbPipeline([
    ('smote', SMOTE(random_state=42)),
    ('model', LogisticRegression(max_iter=1000, random_state=42)),
])
pipe_logreg.fit(X_train, y_train)

y_pred = pipe_logreg.predict(X_test)
y_proba = pipe_logreg.predict_proba(X_test)[:, 1]

m = evaluar(y_test, y_pred, y_proba)
m.update({'modelo': 'LogisticRegression', 'estrategia': 'C · SMOTE'})
resultados.append(m)
m

{'accuracy': 0.8136465230042592,
 'precision': 0.775243081525804,
 'recall': 0.7039049235993209,
 'f1': 0.7378537106246663,
 'roc_auc': 0.8900835377691395,
 'modelo': 'LogisticRegression',
 'estrategia': 'C · SMOTE'}

In [13]:
# --- XGBoost + SMOTE ---
pipe_xgb = ImbPipeline([
    ('smote', SMOTE(random_state=42)),
    ('model', XGBClassifier(
        n_estimators=300, max_depth=8, learning_rate=0.1,
        eval_metric='logloss', n_jobs=-1, random_state=42,
    )),
])
pipe_xgb.fit(X_train, y_train)

y_pred = pipe_xgb.predict(X_test)
y_proba = pipe_xgb.predict_proba(X_test)[:, 1]

m = evaluar(y_test, y_pred, y_proba)
m.update({'modelo': 'XGBoost', 'estrategia': 'C · SMOTE'})
resultados.append(m)
m

{'accuracy': 0.8735714586935436,
 'precision': 0.8462451061810417,
 'recall': 0.8073571024335031,
 'f1': 0.8263438368860055,
 'roc_auc': 0.946318455184225,
 'modelo': 'XGBoost',
 'estrategia': 'C · SMOTE'}

## 6. Comparativa

Reunimos todos los resultados en una tabla ordenada y los visualizamos.

In [14]:
# Tabla ordenada (modelo, estrategia, métricas)
df_resultados = pd.DataFrame(resultados)[
    ['modelo', 'estrategia', 'accuracy', 'precision', 'recall', 'f1', 'roc_auc']
]
df_resultados = df_resultados.sort_values(['modelo', 'estrategia']).reset_index(drop=True)
df_resultados.round(4)

,modelo,estrategia,accuracy,precision,recall,f1,roc_auc
0,LogisticRegression,A · Baseline,0.8119,0.8067,0.6510,0.7206,0.8916
1,LogisticRegression,B · class_weight,0.8090,0.7270,0.7804,0.7528,0.8917
2,LogisticRegression,C · SMOTE,0.8136,0.7752,0.7039,0.7379,0.8901
3,XGBoost,A · Baseline,0.8737,0.8576,0.7928,0.8239,0.9467
4,XGBoost,B · scale_pos_weight,0.8696,0.8097,0.8496,0.8292,0.9467
5,XGBoost,C · SMOTE,0.8736,0.8462,0.8074,0.8263,0.9463


In [15]:
# Comparamos recall, precision y roc_auc por estrategia (gráfico agrupado)
df_largo = df_resultados.melt(
    id_vars=['modelo', 'estrategia'],
    value_vars=['recall', 'precision', 'roc_auc'],
    var_name='metrica', value_name='valor',
)

fig = px.bar(
    df_largo,
    x='estrategia',
    y='valor',
    color='metrica',
    barmode='group',
    facet_col='modelo',
    title='Efecto del balanceo: recall ↑, precision ↓, ROC-AUC ~igual',
    labels={'estrategia': 'Estrategia', 'valor': 'Valor', 'metrica': 'Métrica'},
    width=950, height=480,
)
fig.update_yaxes(range=[0, 1])
fig.update_xaxes(tickangle=-20)
fig.show(renderer='notebook_connected')

**Lectura del gráfico.**

- Con balanceo (B y C), el **recall sube** (detectamos más cancelaciones)… pero la **precision baja** (a costa de más falsos positivos). Es el clásico *trade-off*.
- El **ROC-AUC apenas se mueve**: ordenar bien las reservas por probabilidad no depende de cuántos positivos haya en *train*, así que balancear **no mejora** esta métrica.
- XGBoost domina a la regresión logística en todas las estrategias.

## 7. Conclusión

- El balanceo (`class_weight`, `scale_pos_weight`, SMOTE) **redistribuye el *trade-off* precision/recall** alrededor del umbral 0.5, pero **no mejora el ROC-AUC**, que es independiente del umbral.
- Como en este proyecto **optimizamos ROC-AUC**, el pipeline de producción **no balancea**: el balanceo no aporta nada a la métrica objetivo y sí complica el entrenamiento (más datos sintéticos, otra dependencia).
- El balanceo **solo merece la pena** si lo que te importa es una métrica **dependiente del umbral** (p. ej. maximizar el *recall* de cancelaciones a 0.5). En ese caso, una alternativa más simple y sin tocar el entrenamiento es **mover el umbral** de decisión sobre las probabilidades del modelo baseline.